In [3]:
# CELL 1 — Imports and project paths

from pathlib import Path
import ast
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Find ml-workstream-2 no matter whether Jupyter starts in the project root
# or in its notebooks/ directory.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "data" / "raw").is_dir()
        and (p / "notebooks").is_dir()
        and (p / "reports").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find ml-workstream-2. Expected sibling folders: "
        "data/raw, notebooks, and reports. Current directory: " + str(cwd)
    )

DATASET_NAME = (
    "qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0"
)
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / DATASET_NAME
DATA_DIR = DATASET_ROOT / "data"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

for folder in [INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset data directory not found: {DATA_DIR}")

ECG_FS_HZ = 250.0
PHASE_TO_TARGET = {"relax": 0, "stress": 1}

print("Project root :", PROJECT_ROOT)
print("Raw data     :", DATA_DIR)
print("Processed    :", PROCESSED_DIR)
print("Reports      :", REPORTS_DIR)

Project root : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2
Raw data     : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\raw\qol_stress-a-multimodal-dataset-of-physiological-and-self-reported-stress-responses-1.0.0\data
Processed    : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\data\processed
Reports      : D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports


In [4]:
# CELL 2 — Resolve and load the required collections safely


def find_dataset_file(stem, required=True):
    """Find a dataset file by stem; tolerates hidden/omitted extensions."""
    candidates = [
        DATA_DIR / f"{stem}.csv",
        DATA_DIR / stem,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate

    # Fallback: a single file whose name starts with the requested stem.
    matches = [p for p in DATA_DIR.iterdir() if p.is_file() and p.stem.lower() == stem.lower()]
    if len(matches) == 1:
        return matches[0]
    if required:
        raise FileNotFoundError(
            f"Could not locate {stem}.csv in {DATA_DIR}. "
            f"Available files: {[p.name for p in DATA_DIR.iterdir() if p.is_file()]}"
        )
    return None


def load_collection(stem, required=True):
    path = find_dataset_file(stem, required=required)
    if path is None:
        print(f"Optional collection not found: {stem}")
        return None
    frame = pd.read_csv(path, low_memory=False)
    print(f"Loaded {path.name}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    return frame


ecg_raw = load_collection("ecg", required=True)
eda_raw = load_collection("eda", required=True)
hr_second_raw = load_collection("fitbit_heart_rate_per_second", required=False)
hr_minute_raw = load_collection("fitbit_heart_rate_per_minute", required=False)
stai_raw = load_collection("stai", required=False)
pss_raw = load_collection("pss_14", required=False)
participants_raw = load_collection("participants", required=False)

print("\nECG columns:", list(ecg_raw.columns))
print("EDA columns:", list(eda_raw.columns))

Loaded ecg.csv: 132 rows × 13 columns
Loaded eda.csv: 20,094 rows × 6 columns
Loaded fitbit_heart_rate_per_second.csv: 54,462 rows × 3 columns
Loaded fitbit_heart_rate_per_minute.csv: 2,131 rows × 3 columns
Loaded stai.csv: 132 rows × 24 columns
Loaded pss_14.csv: 66 rows × 17 columns
Loaded participants.csv: 66 rows × 5 columns

ECG columns: ['_id', 'reading_id', 'reading_time', 'wire_id', 'result_classification', 'heart_rate', 'heart_rate_alert', 'firmware_version', 'device_app_version', 'hardware_version', 'username', 'phase', 'waveform_samples']
EDA columns: ['username', 'phase', 'time_sec', 'valid_data', 'activation', 'scl_avg']


In [5]:
# CELL 3 — Canonicalize identifiers and phase labels; inspect expected keys


def locate_column(frame, name):
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    return lookup.get(name.strip().lower())


def require_column(frame, name, collection_name):
    col = locate_column(frame, name)
    if col is None:
        raise KeyError(
            f"Required column '{name}' not found in {collection_name}. "
            f"Available columns: {list(frame.columns)}"
        )
    return col


def normalize_phase(series):
    return series.astype("string").str.strip().str.lower()


ecg_user_col = require_column(ecg_raw, "username", "ecg.csv")
ecg_phase_col = require_column(ecg_raw, "phase", "ecg.csv")
ecg_wave_col = require_column(ecg_raw, "waveform_samples", "ecg.csv")
eda_user_col = require_column(eda_raw, "username", "eda.csv")
eda_phase_col = require_column(eda_raw, "phase", "eda.csv")

# Work on copies; never modify the raw source files.
ecg = ecg_raw.copy()
eda = eda_raw.copy()
ecg["participant_id"] = ecg[ecg_user_col].astype("string").str.strip()
ecg["phase"] = normalize_phase(ecg[ecg_phase_col])
eda["participant_id"] = eda[eda_user_col].astype("string").str.strip()
eda["phase"] = normalize_phase(eda[eda_phase_col])

print("ECG rows:", len(ecg), "| unique participants:", ecg["participant_id"].nunique())
print("EDA rows:", len(eda), "| unique participants:", eda["participant_id"].nunique())
print("\nECG phase counts:")
display(ecg["phase"].value_counts(dropna=False).rename_axis("phase").to_frame("recordings"))
print("EDA phase row counts:")
display(eda["phase"].value_counts(dropna=False).rename_axis("phase").to_frame("rows"))

unexpected_ecg_phases = sorted(set(ecg["phase"].dropna().unique()) - set(PHASE_TO_TARGET))
unexpected_eda_phases = sorted(set(eda["phase"].dropna().unique()) - set(PHASE_TO_TARGET))
if unexpected_ecg_phases or unexpected_eda_phases:
    raise ValueError(
        f"Unexpected phase labels. ECG={unexpected_ecg_phases}; EDA={unexpected_eda_phases}. "
        "Inspect the raw phase values before proceeding."
    )

# The public data description expects one ECG recording per participant and phase.
ecg_key_dupes = ecg.duplicated(["participant_id", "phase"], keep=False)
if ecg_key_dupes.any():
    display(ecg.loc[ecg_key_dupes, ["participant_id", "phase"]].sort_values(["participant_id", "phase"]))
    raise ValueError(
        "Duplicate ECG participant-phase keys detected. Do not merge until these are investigated; "
        "otherwise the feature join could multiply rows."
    )

eda_key_counts = eda.groupby(["participant_id", "phase"], dropna=False).size()
print("\nUnique EDA participant-phase recordings:", int(len(eda_key_counts)))
print("Duplicate EDA participant-phase groups:", int((eda_key_counts > 1).sum()))

ECG rows: 132 | unique participants: 66
EDA rows: 20094 | unique participants: 64

ECG phase counts:


,recordings
phase,
relax,66
stress,66


EDA phase row counts:


,rows
phase,
relax,10367
stress,9727



Unique EDA participant-phase recordings: 124
Duplicate EDA participant-phase groups: 124


In [6]:
# CELL 4 — Parse ECG waveform fields and produce a recording-level quality audit

_NUMBER_PATTERN = re.compile(r"[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?")


def parse_waveform(value):
    """Parse a CSV waveform cell into a 1-D float array; return None if unsupported."""
    if value is None:
        return None
    if isinstance(value, (list, tuple, np.ndarray)):
        parsed = value
    else:
        try:
            missing = pd.isna(value)
            if isinstance(missing, (bool, np.bool_)) and missing:
                return None
        except (TypeError, ValueError):
            pass
        text = str(value).strip()
        if not text:
            return None
        parsed = None
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(text)
                break
            except Exception:
                continue
        if parsed is None:
            # The export may encode a numeric array as comma/semicolon/space-separated text.
            numeric_tokens = _NUMBER_PATTERN.findall(text)
            if not numeric_tokens:
                return None
            try:
                parsed = [float(token) for token in numeric_tokens]
            except ValueError:
                return None
    try:
        arr = np.asarray(parsed, dtype=float).reshape(-1)
        return arr if arr.size else None
    except (ValueError, TypeError, OverflowError):
        return None


ecg["_waveform"] = ecg[ecg_wave_col].map(parse_waveform)

eq_rows = []
for row_idx, row in ecg.iterrows():
    signal = row["_waveform"]
    if isinstance(signal, np.ndarray):
        sample_count = int(signal.size)
        finite_count = int(np.isfinite(signal).sum())
        finite_fraction = finite_count / sample_count if sample_count else np.nan
        signal_std = float(np.nanstd(signal)) if finite_count else np.nan
    else:
        sample_count, finite_count, finite_fraction, signal_std = 0, 0, 0.0, np.nan

    usable = bool(
        isinstance(signal, np.ndarray)
        and sample_count >= 1000
        and finite_fraction >= 0.995
        and np.isfinite(signal_std)
        and signal_std > 1e-12
    )
    eq_rows.append({
        "source_row": row_idx,
        "participant_id": row["participant_id"],
        "phase": row["phase"],
        "ecg_sample_count": sample_count,
        "ecg_finite_sample_count": finite_count,
        "ecg_finite_fraction": finite_fraction,
        "ecg_duration_seconds": sample_count / ECG_FS_HZ if sample_count else np.nan,
        "ecg_signal_std_export_units": signal_std,
        "ecg_signal_usable": usable,
    })

ecg_qc = pd.DataFrame(eq_rows)
print("ECG waveform quality:")
display(ecg_qc[[
    "ecg_sample_count", "ecg_finite_fraction", "ecg_duration_seconds",
    "ecg_signal_std_export_units", "ecg_signal_usable"
]].describe(include="all"))
print("Usable ECG waveforms:", int(ecg_qc["ecg_signal_usable"].sum()), "/", len(ecg_qc))
print("Unparsed waveforms:", int((ecg_qc["ecg_sample_count"] == 0).sum()))

ecg_qc.to_csv(TABLES_DIR / "ecg_waveform_quality_audit.csv", index=False)
display(ecg_qc.head())

ECG waveform quality:


,ecg_sample_count,ecg_finite_fraction,ecg_duration_seconds,ecg_signal_std_export_units,ecg_signal_usable
count,132.000000,132.0,132.000000,132.000000,132
unique,NaN,NaN,NaN,NaN,1
top,NaN,NaN,NaN,NaN,True
freq,NaN,NaN,NaN,NaN,132
mean,7499.886364,1.0,29.999545,1993.643627,NaN
std,1.810266,0.0,0.007241,1109.946542,NaN
min,7488.000000,1.0,29.952000,662.391883,NaN
25%,7500.000000,1.0,30.000000,1244.132246,NaN
50%,7500.000000,1.0,30.000000,1669.341050,NaN
75%,7500.000000,1.0,30.000000,2506.739352,NaN


Usable ECG waveforms: 132 / 132
Unparsed waveforms: 0


,source_row,participant_id,phase,ecg_sample_count,ecg_finite_sample_count,ecg_finite_fraction,ecg_duration_seconds,ecg_signal_std_export_units,ecg_signal_usable
0,0,6683f97240e6fd3620594043bee323169ed91d789c7e99...,relax,7500,7500,1.0,30.000,1808.450233,True
1,1,6683f97240e6fd3620594043bee323169ed91d789c7e99...,stress,7501,7501,1.0,30.004,911.257223,True
2,2,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e1...,relax,7500,7500,1.0,30.000,1362.988333,True
3,3,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e1...,stress,7500,7500,1.0,30.000,1005.323406,True
4,4,29fd79d7bc4d13bd25b1e506705cdd8435f803b2679b1c...,relax,7500,7500,1.0,30.000,1791.599028,True


In [8]:
# CELL 5 — Extract transparent ECG features (no opaque filtering or HRV claim)


def extract_ecg_features(row):
    signal = row["_waveform"]
    base = {
        "participant_id": row["participant_id"],
        "phase": row["phase"],
    }

    device_hr_col = locate_column(ecg, "heart_rate")
    if device_hr_col is not None:
        base["ecg_device_heart_rate_bpm"] = pd.to_numeric(
            pd.Series([row[device_hr_col]]), errors="coerce"
        ).iloc[0]
    else:
        base["ecg_device_heart_rate_bpm"] = np.nan

    if not isinstance(signal, np.ndarray):
        return base

    # Do not silently remove/bridge non-finite ECG samples.
    if not np.isfinite(signal).all() or signal.size < 1000 or np.std(signal) <= 1e-12:
        return base

    x = signal.astype(float)
    x_centered = x - np.median(x)
    q25, q75 = np.percentile(x, [25, 75])
    mad = np.median(np.abs(x - np.median(x)))
    dx = np.diff(x)

    base.update({
        "ecg_mean_export_units": float(np.mean(x)),
        "ecg_median_export_units": float(np.median(x)),
        "ecg_std_export_units": float(np.std(x, ddof=1)),
        "ecg_mad_export_units": float(mad),
        "ecg_iqr_export_units": float(q75 - q25),
        "ecg_q05_export_units": float(np.percentile(x, 5)),
        "ecg_q95_export_units": float(np.percentile(x, 95)),
        "ecg_min_export_units": float(np.min(x)),
        "ecg_max_export_units": float(np.max(x)),
        "ecg_peak_to_peak_export_units": float(np.ptp(x)),
        "ecg_rms_centered_export_units": float(np.sqrt(np.mean(x_centered ** 2))),
        "ecg_abs_first_difference_mean": float(np.mean(np.abs(dx))),
        "ecg_first_difference_std": float(np.std(dx, ddof=1)) if dx.size > 1 else np.nan,
        "ecg_skewness": float(skew(x, bias=False)) if x.size > 2 else np.nan,
        "ecg_excess_kurtosis": float(kurtosis(x, fisher=True, bias=False)) if x.size > 3 else np.nan,
    })

    # Relative spectral power bands; the amplitude unit cancels in the ratios.
    # These are signal descriptors, not diagnostic biomarkers.
    frequencies, power = welch(x_centered, fs=ECG_FS_HZ, nperseg=min(1024, x.size))
    total_mask = (frequencies >= 0.5) & (frequencies <= 40.0)
    total_power = float(np.trapezoid(power[total_mask], frequencies[total_mask])) if total_mask.sum() >= 2 else 0.0
    for band_name, low, high in [
        ("low", 0.5, 5.0),
        ("mid", 5.0, 15.0),
        ("high", 15.0, 40.0),
    ]:
        mask = (frequencies >= low) & (frequencies < high)
        band_power = float(np.trapezoid(power[mask], frequencies[mask])) if mask.sum() >= 2 else np.nan
        base[f"ecg_relative_power_{band_name}_0p5_40hz"] = (
            band_power / total_power if total_power > 0 and np.isfinite(band_power) else np.nan
        )

    return base


ecg_feature_rows = [extract_ecg_features(row) for _, row in ecg.iterrows()]
ecg_features = pd.DataFrame(ecg_feature_rows)
ecg_features = ecg_qc.merge(ecg_features, on=["participant_id", "phase"], how="left", validate="one_to_one")
ecg_features["target"] = ecg_features["phase"].map(PHASE_TO_TARGET)

ecg_features.to_csv(PROCESSED_DIR / "ecg_features.csv", index=False)
print("ECG feature table:", ecg_features.shape)
print("ECG feature columns:", [c for c in ecg_features.columns if c.startswith("ecg_")])
display(ecg_features.head())

ECG feature table: (132, 29)
ECG feature columns: ['ecg_sample_count', 'ecg_finite_sample_count', 'ecg_finite_fraction', 'ecg_duration_seconds', 'ecg_signal_std_export_units', 'ecg_signal_usable', 'ecg_device_heart_rate_bpm', 'ecg_mean_export_units', 'ecg_median_export_units', 'ecg_std_export_units', 'ecg_mad_export_units', 'ecg_iqr_export_units', 'ecg_q05_export_units', 'ecg_q95_export_units', 'ecg_min_export_units', 'ecg_max_export_units', 'ecg_peak_to_peak_export_units', 'ecg_rms_centered_export_units', 'ecg_abs_first_difference_mean', 'ecg_first_difference_std', 'ecg_skewness', 'ecg_excess_kurtosis', 'ecg_relative_power_low_0p5_40hz', 'ecg_relative_power_mid_0p5_40hz', 'ecg_relative_power_high_0p5_40hz']


,source_row,participant_id,phase,ecg_sample_count,ecg_finite_sample_count,ecg_finite_fraction,ecg_duration_seconds,ecg_signal_std_export_units,ecg_signal_usable,ecg_device_heart_rate_bpm,...,ecg_peak_to_peak_export_units,ecg_rms_centered_export_units,ecg_abs_first_difference_mean,ecg_first_difference_std,ecg_skewness,ecg_excess_kurtosis,ecg_relative_power_low_0p5_40hz,ecg_relative_power_mid_0p5_40hz,ecg_relative_power_high_0p5_40hz,target
0,0,6683f97240e6fd3620594043bee323169ed91d789c7e99...,relax,7500,7500,1.0,30.000,1808.450233,True,46,...,25110.0,1823.406022,115.835178,221.025746,4.913665,44.970531,0.542176,0.273031,0.170820,0
1,1,6683f97240e6fd3620594043bee323169ed91d789c7e99...,stress,7501,7501,1.0,30.004,911.257223,True,46,...,10771.0,924.217422,117.381600,225.152094,0.986598,8.327254,0.498976,0.300512,0.183436,1
2,2,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e1...,relax,7500,7500,1.0,30.000,1362.988333,True,78,...,15228.0,1363.547302,143.018536,301.567615,-1.021798,6.157543,0.461374,0.383760,0.145282,0
3,3,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e1...,stress,7500,7500,1.0,30.000,1005.323406,True,71,...,13008.0,1014.937965,103.199360,201.224732,0.372267,5.875526,0.594266,0.306991,0.090400,1
4,4,29fd79d7bc4d13bd25b1e506705cdd8435f803b2679b1c...,relax,7500,7500,1.0,30.000,1791.599028,True,94,...,15097.0,1876.819953,211.261101,555.786692,2.876943,10.874371,0.312156,0.391052,0.252862,0


In [9]:
# CELL 6 — Clean EDA validity flags and aggregate by participant × phase

eda_time_col = require_column(eda, "time_sec", "eda.csv")
eda_valid_col = require_column(eda, "valid_data", "eda.csv")
eda_scl_col = require_column(eda, "scl_avg", "eda.csv")
eda_activation_col = locate_column(eda, "activation")


def parse_bool(series):
    text = series.astype("string").str.strip().str.lower()
    return text.map({
        "true": True, "false": False,
        "1": True, "0": False,
        "1.0": True, "0.0": False,
        "yes": True, "no": False,
    })


eda["_time_sec_numeric"] = pd.to_numeric(eda[eda_time_col], errors="coerce")
eda["_scl_numeric"] = pd.to_numeric(eda[eda_scl_col], errors="coerce")
eda["_valid_bool"] = parse_bool(eda[eda_valid_col])
if eda_activation_col is not None:
    eda["_activation_numeric"] = pd.to_numeric(eda[eda_activation_col], errors="coerce")
else:
    eda["_activation_numeric"] = np.nan

eda_feature_rows = []
for (participant_id, phase), group in eda.groupby(["participant_id", "phase"], dropna=False, sort=True):
    valid_mask = group["_valid_bool"].eq(True) & group["_scl_numeric"].notna()
    valid = group.loc[valid_mask].copy()
    scl = valid["_scl_numeric"].to_numpy(dtype=float)
    times = valid["_time_sec_numeric"].to_numpy(dtype=float)
    time_ok = np.isfinite(times)
    scl_time = scl[time_ok] if len(scl) == len(times) else scl
    time_values = times[time_ok]

    row = {
        "participant_id": participant_id,
        "phase": phase,
        "eda_total_rows": int(len(group)),
        "eda_valid_flag_true_rows": int(group["_valid_bool"].eq(True).sum()),
        "eda_valid_flag_false_rows": int(group["_valid_bool"].eq(False).sum()),
        "eda_valid_flag_unknown_rows": int(group["_valid_bool"].isna().sum()),
        "eda_valid_scl_rows": int(len(scl)),
        "eda_scl_valid_fraction": float(len(valid) / len(group)) if len(group) else np.nan,
        "eda_unique_time_markers": int(group["_time_sec_numeric"].nunique(dropna=True)),
        "eda_repeated_time_rows": int(group["_time_sec_numeric"].duplicated(keep=False).sum()),
        "eda_recording_span_seconds": (
            float(group["_time_sec_numeric"].max() - group["_time_sec_numeric"].min())
            if group["_time_sec_numeric"].notna().any() else np.nan
        ),
    }

    if len(scl):
        row.update({
            "eda_scl_mean_us": float(np.mean(scl)),
            "eda_scl_median_us": float(np.median(scl)),
            "eda_scl_std_us": float(np.std(scl, ddof=1)) if len(scl) > 1 else np.nan,
            "eda_scl_mad_us": float(np.median(np.abs(scl - np.median(scl)))),
            "eda_scl_iqr_us": float(np.percentile(scl, 75) - np.percentile(scl, 25)),
            "eda_scl_min_us": float(np.min(scl)),
            "eda_scl_max_us": float(np.max(scl)),
        })
    else:
        for col in [
            "eda_scl_mean_us", "eda_scl_median_us", "eda_scl_std_us",
            "eda_scl_mad_us", "eda_scl_iqr_us", "eda_scl_min_us", "eda_scl_max_us"
        ]:
            row[col] = np.nan

    # Trend uses the original time_sec markers. Repeated timestamps are retained;
    # no 1-Hz resampling or interpolation is performed.
    if len(time_values) >= 3 and np.unique(time_values).size >= 2:
        row["eda_scl_slope_us_per_sec"] = float(np.polyfit(time_values, scl_time, 1)[0])
    else:
        row["eda_scl_slope_us_per_sec"] = np.nan

    # Kept for auditing only; excluded from the first physiological feature set
    # because activation is a device-provided derived indicator.
    activation_values = pd.to_numeric(group["_activation_numeric"], errors="coerce")
    row["eda_device_activation_mean_audit_only"] = (
        float(activation_values.mean()) if activation_values.notna().any() else np.nan
    )

    eda_feature_rows.append(row)

eda_features = pd.DataFrame(eda_feature_rows)
eda_features["target"] = eda_features["phase"].map(PHASE_TO_TARGET)
eda_features.to_csv(PROCESSED_DIR / "eda_features.csv", index=False)

print("EDA participant-phase feature table:", eda_features.shape)
print("Rows with no valid SCL measurements:", int(eda_features["eda_valid_scl_rows"].eq(0).sum()))
display(eda_features.head())

EDA participant-phase feature table: (124, 21)
Rows with no valid SCL measurements: 0


,participant_id,phase,eda_total_rows,eda_valid_flag_true_rows,eda_valid_flag_false_rows,eda_valid_flag_unknown_rows,eda_valid_scl_rows,eda_scl_valid_fraction,eda_unique_time_markers,eda_repeated_time_rows,...,eda_scl_mean_us,eda_scl_median_us,eda_scl_std_us,eda_scl_mad_us,eda_scl_iqr_us,eda_scl_min_us,eda_scl_max_us,eda_scl_slope_us_per_sec,eda_device_activation_mean_audit_only,target
0,02b865b4b8a0f6319640b266de23bd0cfdffc345e75ec6...,relax,163,160,3,0,160,0.981595,33,163,...,27.182062,26.780,1.094407,0.450,1.2175,26.09,31.68,-0.019852,0.159509,0
1,036e6da345803c97c92d64a489ecdfefd93adb7cd94743...,relax,163,160,3,0,160,0.981595,33,163,...,38.015812,37.950,0.632001,0.355,0.5375,35.95,41.18,-0.007423,0.122699,0
2,036e6da345803c97c92d64a489ecdfefd93adb7cd94743...,stress,163,160,3,0,160,0.981595,33,163,...,41.635625,41.745,0.968670,0.595,1.5500,40.22,45.68,-0.019181,0.208589,1
3,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e1...,relax,163,159,4,0,159,0.975460,33,163,...,32.541824,32.390,0.934719,0.560,1.0250,30.92,36.56,-0.015159,0.153374,0
4,06c274b06c4badedf4a5710f3af21a338af00c3ac3e6e1...,stress,162,160,2,0,160,0.987654,33,162,...,33.994813,33.720,1.084662,0.450,0.8425,32.72,38.62,-0.019143,0.129630,1


In [10]:
# CELL 7 — Summarize Fitbit heart rate separately (do not invent phase labels)


def summarize_fitbit_hr(frame, resolution_name):
    if frame is None or frame.empty:
        print(f"No Fitbit {resolution_name} collection available.")
        return None

    user_col = locate_column(frame, "username")
    value_col = locate_column(frame, "value")
    if user_col is None or value_col is None:
        print(f"Skipping Fitbit {resolution_name}: expected username/value columns not found.")
        return None

    work = frame.copy()
    work["participant_id"] = work[user_col].astype("string").str.strip()
    work["heart_rate_bpm"] = pd.to_numeric(work[value_col], errors="coerce")
    summary = (
        work.groupby("participant_id")["heart_rate_bpm"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )
    summary = summary.rename(columns={
        "count": f"{resolution_name}_count",
        "mean": f"{resolution_name}_mean_bpm",
        "median": f"{resolution_name}_median_bpm",
        "std": f"{resolution_name}_std_bpm",
        "min": f"{resolution_name}_min_bpm",
        "max": f"{resolution_name}_max_bpm",
    })
    summary.to_csv(PROCESSED_DIR / f"fitbit_{resolution_name}_participant_summary_audit_only.csv", index=False)
    print(f"Fitbit {resolution_name} participant summaries:", summary.shape)
    return summary


fitbit_second_summary = summarize_fitbit_hr(hr_second_raw, "hr_per_second")
fitbit_minute_summary = summarize_fitbit_hr(hr_minute_raw, "hr_per_minute")

print(
    "\nThese Fitbit summaries are participant-level across the available session, "
    "not phase-level. They are saved for later temporal-alignment work and are "
    "not merged into the RELAX/STRESS training table."
)

Fitbit hr_per_second participant summaries: (66, 7)
Fitbit hr_per_minute participant summaries: (66, 7)

These Fitbit summaries are participant-level across the available session, not phase-level. They are saved for later temporal-alignment work and are not merged into the RELAX/STRESS training table.


In [11]:
# CELL 8 — Keep questionnaires and demographics as separate reference tables

# STAI has phase-level self-report scores. It is useful for later construct
# comparison, but it is not an input feature for physiology-only phase prediction.
if stai_raw is not None:
    stai_user_col = locate_column(stai_raw, "username")
    stai_phase_col = locate_column(stai_raw, "phase")
    stai_score_col = locate_column(stai_raw, "score")
    if stai_user_col is not None and stai_phase_col is not None:
        stai_reference = stai_raw.copy()
        stai_reference["participant_id"] = stai_reference[stai_user_col].astype("string").str.strip()
        stai_reference["phase"] = normalize_phase(stai_reference[stai_phase_col])
        if stai_score_col is not None:
            stai_reference["stai_score_numeric"] = pd.to_numeric(stai_reference[stai_score_col], errors="coerce")
        stai_reference.to_csv(INTERIM_DIR / "stai_reference_not_model_input.csv", index=False)
        print("Saved STAI reference table separately:", stai_reference.shape)
    else:
        print("STAI file found, but username/phase columns were not both present; no reference join made.")

if pss_raw is not None:
    pss_raw.to_csv(INTERIM_DIR / "pss14_reference_not_model_input.csv", index=False)
    print("Saved PSS-14 reference table separately:", pss_raw.shape)

if participants_raw is not None:
    participants_raw.to_csv(INTERIM_DIR / "participant_demographics_reference_not_model_input.csv", index=False)
    print("Saved participant demographics separately:", participants_raw.shape)

print("Questionnaire scores and demographics are intentionally excluded from the first physiological model inputs.")

Saved STAI reference table separately: (132, 26)
Saved PSS-14 reference table separately: (66, 17)
Saved participant demographics separately: (66, 5)
Questionnaire scores and demographics are intentionally excluded from the first physiological model inputs.


In [12]:
# CELL 9 — Build the participant-phase dataset and run leakage/merge checks

# ECG is expected to provide one row per participant-phase. EDA may be absent
# for some participant-phase recordings; an outer join preserves those rows.
combined = ecg_features.merge(
    eda_features,
    on=["participant_id", "phase", "target"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)
combined["eda_recording_present"] = combined["_merge"].isin(["both", "right_only"])
combined = combined.drop(columns=["_merge"])
combined["target"] = combined["phase"].map(PHASE_TO_TARGET).astype("Int64")

# Participant ID and phase are grouping/label columns, never model features.
# Exclude recording-quality counters and the device-derived EDA activation indicator.
ECG_MODEL_FEATURES = [
    c for c in combined.columns
    if c.startswith("ecg_")
    and c not in {
        "ecg_signal_usable",
        "ecg_sample_count",
        "ecg_finite_sample_count",
        "ecg_finite_fraction",
        "ecg_duration_seconds",
        "ecg_signal_std_export_units",
    }
]
EDA_MODEL_FEATURES = [
    c for c in [
        "eda_scl_mean_us",
        "eda_scl_median_us",
        "eda_scl_std_us",
        "eda_scl_mad_us",
        "eda_scl_iqr_us",
        "eda_scl_min_us",
        "eda_scl_max_us",
        "eda_scl_slope_us_per_sec",
    ]
    if c in combined.columns
]
MODEL_FEATURES = ECG_MODEL_FEATURES + EDA_MODEL_FEATURES

# Include a transparent numeric target and preserve missing feature values.
combined.to_csv(PROCESSED_DIR / "participant_phase_features.csv", index=False)

participant_manifest = (
    combined[["participant_id"]]
    .dropna()
    .drop_duplicates()
    .sort_values("participant_id")
    .reset_index(drop=True)
)
participant_manifest["group_id"] = np.arange(len(participant_manifest), dtype=int)
participant_manifest.to_csv(PROCESSED_DIR / "participant_group_manifest.csv", index=False)

print("Combined participant-phase table:", combined.shape)
print("Unique participants:", combined["participant_id"].nunique())
print("Phase counts:")
display(combined["phase"].value_counts(dropna=False).rename_axis("phase").to_frame("rows"))
print("\nRows with absent EDA recording:", int((~combined["eda_recording_present"]).sum()))
print("Candidate physiological model features:")
for feature in MODEL_FEATURES:
    print(" -", feature)

# Guardrails before model development.
if combined.duplicated(["participant_id", "phase"]).any():
    raise AssertionError("Combined feature table contains duplicate participant-phase rows.")
if combined["target"].isna().any():
    raise AssertionError("Some rows have no target label; inspect phase values.")
if "participant_id" in MODEL_FEATURES or "phase" in MODEL_FEATURES or "target" in MODEL_FEATURES:
    raise AssertionError("Identifier/label leakage detected in MODEL_FEATURES.")
if any("activation" in c for c in MODEL_FEATURES):
    raise AssertionError("Device-derived EDA activation unexpectedly entered MODEL_FEATURES.")

missingness = (
    combined[MODEL_FEATURES]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)
missingness.to_csv(TABLES_DIR / "model_feature_missingness.csv", index=False)
display(missingness)

Combined participant-phase table: (132, 48)
Unique participants: 66
Phase counts:


,rows
phase,
relax,66
stress,66



Rows with absent EDA recording: 8
Candidate physiological model features:
 - ecg_device_heart_rate_bpm
 - ecg_mean_export_units
 - ecg_median_export_units
 - ecg_std_export_units
 - ecg_mad_export_units
 - ecg_iqr_export_units
 - ecg_q05_export_units
 - ecg_q95_export_units
 - ecg_min_export_units
 - ecg_max_export_units
 - ecg_peak_to_peak_export_units
 - ecg_rms_centered_export_units
 - ecg_abs_first_difference_mean
 - ecg_first_difference_std
 - ecg_skewness
 - ecg_excess_kurtosis
 - ecg_relative_power_low_0p5_40hz
 - ecg_relative_power_mid_0p5_40hz
 - ecg_relative_power_high_0p5_40hz
 - eda_scl_mean_us
 - eda_scl_median_us
 - eda_scl_std_us
 - eda_scl_mad_us
 - eda_scl_iqr_us
 - eda_scl_min_us
 - eda_scl_max_us
 - eda_scl_slope_us_per_sec


,feature,missing_fraction
0,eda_scl_min_us,0.060606
1,eda_scl_max_us,0.060606
2,eda_scl_slope_us_per_sec,0.060606
3,eda_scl_mean_us,0.060606
4,eda_scl_median_us,0.060606
5,eda_scl_std_us,0.060606
6,eda_scl_mad_us,0.060606
7,eda_scl_iqr_us,0.060606
8,ecg_device_heart_rate_bpm,0.000000
9,ecg_mean_export_units,0.000000


In [13]:
# CELL 10 — Save QC tables and a preprocessing report

qc_summary = pd.DataFrame([
    {"check": "ECG recordings", "observed": int(len(ecg)), "reference_expected": 132},
    {"check": "Unique participants in ECG", "observed": int(ecg["participant_id"].nunique()), "reference_expected": 66},
    {"check": "Usable ECG waveforms", "observed": int(ecg_qc["ecg_signal_usable"].sum()), "reference_expected": "Not specified"},
    {"check": "EDA raw rows", "observed": int(len(eda)), "reference_expected": "Row count varies by recording"},
    {"check": "EDA participant-phase recordings", "observed": int(len(eda_features)), "reference_expected": 124},
    {"check": "Combined participant-phase rows", "observed": int(len(combined)), "reference_expected": 132},
    {"check": "Combined unique participants", "observed": int(combined["participant_id"].nunique()), "reference_expected": 66},
    {"check": "Rows missing EDA recording", "observed": int((~combined["eda_recording_present"]).sum()), "reference_expected": 8},
])
qc_summary.to_csv(TABLES_DIR / "preprocessing_qc_summary.csv", index=False)
display(qc_summary)

feature_missing = combined[MODEL_FEATURES].isna().mean().sort_values(ascending=False)

report_lines = [
    "# Preprocessing and Feature Engineering Report",
    "",
    f"- Project root: `{PROJECT_ROOT}`",
    f"- Dataset: `{DATASET_NAME}`",
    f"- ECG recordings loaded: {len(ecg)}",
    f"- Usable ECG waveforms by the documented quality gate: {int(ecg_qc['ecg_signal_usable'].sum())}",
    f"- EDA participant-phase aggregates: {len(eda_features)}",
    f"- Combined participant-phase rows: {len(combined)}",
    f"- Unique participants: {combined['participant_id'].nunique()}",
    f"- Rows without an EDA recording: {int((~combined['eda_recording_present']).sum())}",
    "",
    "## Outputs",
    "- `data/processed/ecg_features.csv`",
    "- `data/processed/eda_features.csv`",
    "- `data/processed/participant_phase_features.csv`",
    "- `data/processed/participant_group_manifest.csv`",
    "- `data/processed/fitbit_hr_per_second_participant_summary_audit_only.csv` (when available)",
    "- `data/processed/fitbit_hr_per_minute_participant_summary_audit_only.csv` (when available)",
    "- `reports/tables/ecg_waveform_quality_audit.csv`",
    "- `reports/tables/model_feature_missingness.csv`",
    "- `reports/tables/preprocessing_qc_summary.csv`",
    "",
    "## Candidate model features",
] + [f"- `{c}`" for c in MODEL_FEATURES] + [
    "",
    "## Feature missingness",
] + [f"- `{feature}`: {fraction:.1%}" for feature, fraction in feature_missing.items()] + [
    "",
    "## Methodological safeguards",
    "- Raw data were not overwritten.",
    "- ECG waveforms with too few samples, non-finite sample fraction above 0.5%, or near-zero variance were not assigned waveform summary/spectral features.",
    "- ECG amplitude features are in unspecified exported units; no conversion to mV was assumed.",
    "- No R-peak/HRV claims were made from these approximately 30-second recordings.",
    "- EDA SCL summaries use only rows marked `valid_data=True`; original `time_sec` markers and repeated timestamps were retained.",
    "- Fitbit heart-rate summaries were not assigned phase labels because the documented files do not contain explicit phase labels.",
    "- STAI/PSS and demographics were retained separately, not used as physiological model inputs.",
    "- Missing values were preserved; imputation and scaling must be fitted inside training folds in the next notebook.",
    "- Validation must be participant-independent (grouped/LOSO); random row splitting would leak participant-specific patterns.",
    "",
    "## Interpretation boundary",
    "This feature table supports an exploratory baseline for classifying the controlled RELAX/STRESS experimental phase in healthy adults. It is not a validated stress diagnosis, clinical decision tool, or validated astronaut-health model.",
    "",
    "## Dataset citation",
    "Calvo, A., Cejudo, A., & Martín Andonegui, C. (2026). *QoL_Stress: A Multimodal Dataset of Physiological and Self-Reported Stress Responses* (version 1.0.0). PhysioNet. https://doi.org/10.13026/vqtz-pm70",
]

report_path = REPORTS_DIR / "preprocessing_feature_engineering_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("\nSaved report:", report_path)
print("\nPreprocessing and feature engineering completed.")

,check,observed,reference_expected
0,ECG recordings,132,132
1,Unique participants in ECG,66,66
2,Usable ECG waveforms,132,Not specified
3,EDA raw rows,20094,Row count varies by recording
4,EDA participant-phase recordings,124,124
5,Combined participant-phase rows,132,132
6,Combined unique participants,66,66
7,Rows missing EDA recording,8,8



Saved report: D:\Projects\TakeOFF\astronaut_health_spec_v0_2\astronaut_health_spec\ml-workstream-2\reports\preprocessing_feature_engineering_report.md

Preprocessing and feature engineering completed.
